# TPV Forecast v2 — Production Training

Trains the **GetTPVForecast v2** (Huber, log-space) model for all four MCCs in the dataset
and writes artifacts directly to `ml_service/artifacts/tpv/` so the live service can hot-reload them.

**What this fixes:**  
The existing artifacts were trained on composite KNN merchants (~$100/month) so the model
extrapolates wildly outside its training distribution at production scale (~$200k/month).
This notebook aggregates the raw transaction data, trains on the actual merchant population,
and produces scale-appropriate artifacts with a meaningful `global_q90`.

**Input:** `data/processed_transactions_4mcc.csv` (raw transaction-level, 2010–2019)  
**Output:** `ml_service/artifacts/tpv/{mcc}/{ctx_len}/` — 8 pkl files + `config_snapshot.json`  
**MCCs trained:** 5411, 5499, 5812, 4121  
**Context lengths:** 1, 3, 6 months  
**Estimated runtime:** 15–60 min depending on hardware (runs locally overnight is fine)

In [ ]:
import subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _imp in [("pandas", "pandas"), ("numpy", "numpy"),
                    ("scikit-learn", "sklearn"), ("joblib", "joblib")]:
    _ensure(_pkg, _imp)

import json
import math
import os
import shutil
import tempfile
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print("Imports OK")

## Configuration

`ARTIFACTS_OUTPUT_PATH` points to the live `ml_service/artifacts/tpv/` folder.  
The running container mounts this directory and **hot-reloads within 60 s** when `config_snapshot.json` mtimes change.

Set `USE_FULL_DATA = True` because the dataset covers 2010–2019 (older than the default 3-year rolling window).

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = Path(os.getcwd())                          # ml_pipeline/
REPO_ROOT      = NOTEBOOK_DIR.parent                        # 404_found_us/
RAW_DATA_PATH  = REPO_ROOT / "data" / "processed_transactions_4mcc.csv"
ARTIFACTS_OUTPUT_PATH = REPO_ROOT / "ml_service" / "artifacts" / "tpv"

# ── Which MCCs to train ────────────────────────────────────────────────────
# All four MCCs present in the dataset. The ml_service will fall back to
# SARIMA for any MCC not in this list, so adding them all is safe.
TRAIN_MCCS = [5411, 5499, 5812, 4121]

# ── Pipeline constants (must match ml_service/modules/tpv_forecast/service.py)
SUPPORTED_CONTEXT_LENS = [1, 3, 6]
HORIZON_LEN   = 3
TARGET_COV    = 0.90
MIN_POOL      = 10
KNN_K         = 10
TARGET_COL    = "total_processing_value"
LOG_TARGET    = "log_tpv"
_EPS          = 1e-6

# ── Use the full dataset instead of a rolling 3-year window ────────────────
# The raw data covers 2010–2019. A 3-year window from today (2026) would
# return zero rows, so we use everything.
USE_FULL_DATA = True

# ── GBR risk model hyper-parameters ───────────────────────────────────────
GBR_N_ESTIMATORS   = 120
GBR_LEARNING_RATE  = 0.05
GBR_MAX_DEPTH      = 2
GBR_MIN_SAMPLES_LEAF = max(20, MIN_POOL)
GBR_SUBSAMPLE      = 0.8
GBR_RANDOM_STATE   = 4121

VOL_BUCKET_SCHEMES = {
    "low-mid-high_50_85":          [0.00, 0.50, 0.85, 1.00],
    "low-mid-high_40_80":          [0.00, 0.40, 0.80, 1.00],
    "low-mid-high_60_90":          [0.00, 0.60, 0.90, 1.00],
    "low-mid-high-vhigh_50_75_90": [0.00, 0.50, 0.75, 0.90, 1.00],
    "low-mid-high-vhigh_40_70_88": [0.00, 0.40, 0.70, 0.88, 1.00],
    "low-mid-high-vhigh_60_85_95": [0.00, 0.60, 0.85, 0.95, 1.00],
}
VOL_MIN_GAIN_ABS  = 0.05
VOL_MIN_GAIN_REL  = 0.01
VOL_TEST_COV_SLACK = 0.03

# ── Re-run control ─────────────────────────────────────────────────────────
# Set True to overwrite MCCs that already have complete artifact sets.
FORCE_RETRAIN = False

print(f"Repo root:        {REPO_ROOT}")
print(f"Raw data:         {RAW_DATA_PATH}")
print(f"Artifacts output: {ARTIFACTS_OUTPUT_PATH}")
print(f"MCCs to train:    {TRAIN_MCCS}")
print(f"FORCE_RETRAIN:    {FORCE_RETRAIN}")


## Step 1 — Load raw transaction data

Load `processed_transactions_4mcc.csv` and display a summary per MCC.

In [ ]:
print(f"Loading {RAW_DATA_PATH} ...")
raw_df = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
raw_df["amount"] = pd.to_numeric(raw_df["amount"], errors="coerce").fillna(0.0)
raw_df["cost_type_ID"] = pd.to_numeric(raw_df["cost_type_ID"], errors="coerce")

print(f"Shape: {raw_df.shape}")
print(f"\nPer-MCC summary:")
summary = (
    raw_df.groupby("mcc")
    .agg(
        rows=("transaction_id", "count"),
        merchants=("merchant_id", "nunique"),
        year_min=("year", "min"),
        year_max=("year", "max"),
        total_volume=("amount", "sum"),
    )
    .assign(avg_monthly_per_merchant=lambda d: d["total_volume"] / d["merchants"] / 120)
)
print(summary.to_string())

## Step 2 — Aggregate to monthly summaries

For each `(merchant_id, year, month)` compute:
- `total_processing_value` — sum of transaction amounts  
- `transaction_count`, `avg_transaction_value`, `std_txn_amount`, `median_txn_amount`  
- `cost_type_X_pct` — share of each cost_type_ID within the month  

This produces the monthly-level DataFrame that the v2 training pipeline expects.

In [ ]:
def aggregate_to_monthly(txn_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert transaction-level rows to merchant×month summaries.

    Input columns required: merchant_id, date (datetime), amount, cost_type_ID
    Returns one row per (merchant_id, year, month).
    """
    df = txn_df.copy()
    df["year"]  = df["date"].dt.year
    df["month"] = df["date"].dt.month

    # ── Core monthly stats ────────────────────────────────────────────────
    agg = (
        df.groupby(["merchant_id", "year", "month"])["amount"]
        .agg(
            total_processing_value="sum",
            transaction_count="count",
            avg_transaction_value="mean",
            std_txn_amount="std",
            median_txn_amount="median",
        )
        .reset_index()
    )
    agg["std_txn_amount"] = agg["std_txn_amount"].fillna(0.0)

    # ── cost_type_X_pct pivot ─────────────────────────────────────────────
    ct_valid = df.dropna(subset=["cost_type_ID"]).copy()
    ct_valid["cost_type_ID"] = ct_valid["cost_type_ID"].astype(int).astype(str)

    ct_counts = (
        ct_valid.groupby(["merchant_id", "year", "month", "cost_type_ID"])
        .size()
        .reset_index(name="ct_count")
    )
    ct_totals = ct_counts.groupby(["merchant_id", "year", "month"])["ct_count"].transform("sum")
    ct_counts["ct_pct"] = ct_counts["ct_count"] / ct_totals

    ct_pivot = ct_counts.pivot_table(
        index=["merchant_id", "year", "month"],
        columns="cost_type_ID",
        values="ct_pct",
        fill_value=0.0,
    ).reset_index()
    ct_pivot.columns = [
        f"cost_type_{c}_pct" if c not in ("merchant_id", "year", "month") else c
        for c in ct_pivot.columns
    ]

    monthly = agg.merge(ct_pivot, on=["merchant_id", "year", "month"], how="left")
    ct_cols = [c for c in monthly.columns if c.startswith("cost_type_") and c.endswith("_pct")]
    monthly[ct_cols] = monthly[ct_cols].fillna(0.0)

    # Add log target
    monthly[LOG_TARGET] = np.log1p(monthly[TARGET_COL].astype(float))

    return monthly.sort_values(["merchant_id", "year", "month"]).reset_index(drop=True)


# Aggregate all MCCs and store in a dict for quick lookup
print("Aggregating transactions to monthly summaries...")
monthly_by_mcc: Dict[int, pd.DataFrame] = {}
for mcc in TRAIN_MCCS:
    t0 = time.monotonic()
    mcc_txn = raw_df[raw_df["mcc"] == mcc]
    monthly = aggregate_to_monthly(mcc_txn)
    monthly_by_mcc[mcc] = monthly
    ct_cols = [c for c in monthly.columns if c.startswith("cost_type_")]
    print(
        f"  MCC {mcc}: {len(monthly):,} merchant-months, "
        f"{monthly['merchant_id'].nunique():,} merchants, "
        f"{len(ct_cols)} cost_type cols  [{time.monotonic()-t0:.1f}s]"
    )

print("\nDone.")

## Step 3 — Training infrastructure

Inline implementation of all helper functions from `GetTPVForecast Service v2/train.py`.  
No external imports needed — runs standalone.

In [ ]:
# ── Scenario generation ────────────────────────────────────────────────────

def _build_scenarios(df: pd.DataFrame, context_len: int) -> List[dict]:
    scenarios = []
    for mid, mdf in df.groupby("merchant_id"):
        mdf = mdf.sort_values(["year", "month"]).reset_index(drop=True)
        n = len(mdf)
        for i in range(n - context_len - HORIZON_LEN + 1):
            ctx = mdf.iloc[i : i + context_len].copy().reset_index(drop=True)
            hor = mdf.iloc[i + context_len : i + context_len + HORIZON_LEN].copy().reset_index(drop=True)

            ctx_span = (
                (ctx.iloc[-1]["year"] - ctx.iloc[0]["year"]) * 12
                + ctx.iloc[-1]["month"] - ctx.iloc[0]["month"]
            )
            if ctx_span != context_len - 1:
                continue

            ctx_end_yr = int(ctx.iloc[-1]["year"])
            ctx_end_mo = int(ctx.iloc[-1]["month"])
            expected_yr = ctx_end_yr + (1 if ctx_end_mo == 12 else 0)
            expected_mo = 1 if ctx_end_mo == 12 else ctx_end_mo + 1

            if int(hor.iloc[0]["year"]) != expected_yr or int(hor.iloc[0]["month"]) != expected_mo:
                continue

            hor_span = (
                (hor.iloc[-1]["year"] - hor.iloc[0]["year"]) * 12
                + hor.iloc[-1]["month"] - hor.iloc[0]["month"]
            )
            if hor_span != HORIZON_LEN - 1:
                continue

            scenarios.append({
                "merchant_id": mid,
                "context_data": ctx,
                "horizon_data": hor,
                "context_range": (
                    (int(ctx.iloc[0]["year"]), int(ctx.iloc[0]["month"])),
                    (ctx_end_yr, ctx_end_mo),
                ),
                "horizon_range": (
                    (expected_yr, expected_mo),
                    (int(hor.iloc[-1]["year"]), int(hor.iloc[-1]["month"])),
                ),
            })
    return scenarios


# ── Merchant-level 60/20/20 split ─────────────────────────────────────────

def _merchant_split(
    scenarios: List[dict], seed: int = 42,
) -> Tuple[List[dict], List[dict], List[dict]]:
    rng = np.random.default_rng(seed)
    all_mids = sorted({s["merchant_id"] for s in scenarios})
    perm = rng.permutation(len(all_mids))
    n = len(all_mids)
    n_train = int(0.60 * n)
    n_val   = int(0.20 * n)
    train_mids = {all_mids[i] for i in perm[:n_train]}
    val_mids   = {all_mids[i] for i in perm[n_train : n_train + n_val]}
    return (
        [s for s in scenarios if s["merchant_id"] in train_mids],
        [s for s in scenarios if s["merchant_id"] in val_mids],
        [s for s in scenarios if s["merchant_id"] not in train_mids | val_mids],
    )


# ── kNN pool-mean cache ────────────────────────────────────────────────────

def _build_pool_caches(
    df: pd.DataFrame,
    scenarios: List[dict],
    cost_type_cols: List[str],
) -> Tuple[Dict[tuple, float], Dict[tuple, float]]:
    unique_keys = {
        (
            s["merchant_id"],
            int(s["context_data"].iloc[-1]["year"]),
            int(s["context_data"].iloc[-1]["month"]),
        )
        for s in scenarios
    }
    print(f"  Building flat pool mean cache ({len(unique_keys):,} keys)...")
    flat_cache: Dict[tuple, float] = {}
    for mid, yr, mo in unique_keys:
        snap = df[
            (df["merchant_id"] != mid)
            & ((df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo)))
        ]
        flat_cache[(mid, yr, mo)] = float(snap[LOG_TARGET].mean()) if len(snap) > 0 else 0.0

    knn_cache: Dict[tuple, float] = dict(flat_cache)
    if not cost_type_cols:
        print("  No cost_type columns — flat pool mean used for kNN cache.")
        return flat_cache, knn_cache

    keys_by_date: Dict[tuple, list] = defaultdict(list)
    for mid, yr, mo in unique_keys:
        keys_by_date[(yr, mo)].append(mid)

    n_dates = len(keys_by_date)
    print(f"  Building kNN pool mean cache ({len(unique_keys):,} keys, {n_dates} dates, k={KNN_K})...")

    for idx, ((yr, mo), query_mids) in enumerate(sorted(keys_by_date.items())):
        snap = df[(df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo))]
        fp_all = snap.groupby("merchant_id")[cost_type_cols].mean()
        log_tpv_all = snap.groupby("merchant_id")[LOG_TARGET].mean()

        if len(fp_all) < KNN_K + 1:
            continue

        nn_date = NearestNeighbors(n_neighbors=min(KNN_K + 1, len(fp_all)), metric="cosine")
        nn_date.fit(fp_all.values)
        fp_index = fp_all.index.tolist()

        for mid in query_mids:
            ctx_row = df[
                (df["merchant_id"] == mid) & (df["year"] == yr) & (df["month"] == mo)
            ]
            if len(ctx_row) == 0 or mid not in fp_all.index:
                continue
            _, raw_idx = nn_date.kneighbors(ctx_row[cost_type_cols].values)
            top_ids = [fp_index[i] for i in raw_idx[0] if fp_index[i] != mid][:KNN_K]
            if len(top_ids) == KNN_K:
                knn_cache[(mid, yr, mo)] = float(log_tpv_all.loc[top_ids].mean())

        if (idx + 1) % 20 == 0:
            print(f"    {idx + 1}/{n_dates} dates processed...")

    print("  Pool caches complete.")
    return flat_cache, knn_cache


# ── Feature builders ───────────────────────────────────────────────────────

def _build_feature_matrix(
    scenarios: List[dict], knn_cache: Dict[tuple, float],
) -> np.ndarray:
    rows = []
    for s in scenarios:
        ctx = s["context_data"]
        vals = ctx[LOG_TARGET].values.astype(float)
        c_mean = float(np.mean(vals))
        c_std  = float(np.std(vals))
        mom    = float(vals[-1] - c_mean)
        key    = (s["merchant_id"], int(ctx.iloc[-1]["year"]), int(ctx.iloc[-1]["month"]))
        p_mean = knn_cache.get(key, c_mean)

        txn_amount_std = float(ctx["std_txn_amount"].fillna(0).mean())
        log_txn        = float(np.log1p(ctx["transaction_count"].mean()))
        avg_median_gap = float(np.mean(np.abs(
            ctx["avg_transaction_value"].values - ctx["median_txn_amount"].values
        )))
        last_month     = float(vals[-1])
        log_avg_txn_val = float(np.log1p(ctx["avg_transaction_value"].mean()))

        tc_vals  = np.log1p(ctx["transaction_count"].values.astype(float))
        atv_vals = np.log1p(ctx["avg_transaction_value"].values.astype(float))
        mom_tc   = float(tc_vals[-1]  - np.mean(tc_vals))
        mom_atv  = float(atv_vals[-1] - np.mean(atv_vals))

        rows.append([c_mean, c_std, mom, p_mean,
                     txn_amount_std, log_txn, avg_median_gap,
                     last_month, log_avg_txn_val, mom_tc, mom_atv])
    return np.array(rows, dtype=float)


def _build_risk_features(
    scenarios: List[dict],
    flat_cache: Dict[tuple, float],
    knn_cache: Dict[tuple, float],
) -> np.ndarray:
    rows = []
    for s in scenarios:
        ctx  = s["context_data"]
        vals = ctx[LOG_TARGET].values.astype(float)
        c_mean = float(np.mean(vals))
        _denom = c_mean + _EPS

        avg_txn_val    = float(ctx["avg_transaction_value"].mean())
        intra_txn_cov  = float(ctx["std_txn_amount"].fillna(0).mean()) / (avg_txn_val + _EPS)
        avg_median_gap = float(np.mean(np.abs(
            ctx["avg_transaction_value"].values - ctx["median_txn_amount"].values
        ))) / (avg_txn_val + _EPS)
        log_txn_count  = float(np.log1p(ctx["transaction_count"].mean()))

        ct_cols   = [c for c in ctx.columns if c.startswith("cost_type_") and c.endswith("_pct")]
        ct_vals_a = ctx[ct_cols].mean().values if ct_cols else np.array([1.0])
        cost_type_hhi = float(np.sum(ct_vals_a ** 2))

        log_avg_txn_val = float(np.log1p(avg_txn_val))
        txn_amount_cov  = float(ctx["std_txn_amount"].fillna(0).mean()) / (avg_txn_val + _EPS)

        yr, mo = s["context_range"][1]
        mid    = s["merchant_id"]
        flat_pm = float(flat_cache.get((mid, yr, mo), c_mean))
        knn_pm  = float(knn_cache.get((mid, yr, mo), c_mean))
        pool_gap = abs(c_mean - flat_pm) / (flat_pm + _EPS)
        knn_gap  = abs(c_mean - knn_pm)  / (knn_pm  + _EPS)
        ctx_cov  = float(np.std(vals)) / _denom

        tc_arr  = np.log1p(ctx["transaction_count"].values.astype(float))
        atv_arr = np.log1p(ctx["avg_transaction_value"].values.astype(float))
        tc_cov  = float(np.std(tc_arr))  / (float(np.mean(tc_arr))  + _EPS)
        atv_cov = float(np.std(atv_arr)) / (float(np.mean(atv_arr)) + _EPS)

        rows.append([
            intra_txn_cov, avg_median_gap, log_txn_count,
            cost_type_hhi, log_avg_txn_val, txn_amount_cov,
            pool_gap, knn_gap, ctx_cov, tc_cov, atv_cov,
        ])
    return np.array(rows, dtype=float)


# ── Conformal helpers ──────────────────────────────────────────────────────

def _adaptive_q(residuals: list, target: float = TARGET_COV) -> Optional[float]:
    if not residuals:
        return None
    n = len(residuals)
    level = math.ceil((n + 1) * target) / n
    return float(np.quantile(residuals, level)) if level <= 1.0 else None


def _make_percentile_bins(ref_vals, apply_vals, pct_edges, min_count=MIN_POOL):
    if len(ref_vals) == 0 or len(apply_vals) == 0:
        return None
    pct_edges = np.array(pct_edges, dtype=float)
    edges = np.quantile(ref_vals, pct_edges)
    edges = np.maximum.accumulate(edges)
    edges = np.unique(edges)
    n_eff = len(edges) - 1
    if n_eff < 2:
        return None
    ref_bins   = np.digitize(ref_vals,   edges[1:-1], right=False)
    apply_bins = np.digitize(apply_vals, edges[1:-1], right=False)
    counts = np.array([(ref_bins == b).sum() for b in range(n_eff)])
    if counts.min() < min_count:
        return None
    return edges, ref_bins, apply_bins, n_eff, counts


def _continuous_width_map(cal_scores_ref, apply_scores, cal_res, pct_edges, q_fallback):
    built = _make_percentile_bins(cal_scores_ref, apply_scores, pct_edges, min_count=MIN_POOL)
    if built is None:
        return None
    edges, ref_bins, apply_bins, n_eff, _ = built
    q_vals = []
    for b in range(n_eff):
        res_b   = cal_res[ref_bins == b].tolist()
        q_local = _adaptive_q(res_b) if len(res_b) >= MIN_POOL else None
        q_vals.append(q_local if q_local is not None else q_fallback)
    q_vals = np.maximum.accumulate(np.array(q_vals, dtype=float))
    knot_x = 0.5 * (edges[:-1] + edges[1:])
    knot_x = np.maximum.accumulate(knot_x + np.arange(len(knot_x)) * 1e-9)
    hw_apply = np.interp(apply_scores, knot_x, q_vals, left=q_vals[0], right=q_vals[-1])
    return {"edges": edges, "knot_x": knot_x, "q_vals": q_vals, "hw_apply": hw_apply}


def _generate_pool_ids(df: pd.DataFrame, merchant_id, yr: int, mo: int) -> set:
    pool = df[
        (df["merchant_id"] != merchant_id)
        & ((df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo)))
    ]
    return set(int(p) for p in pool["merchant_id"].unique())


# ── Atomic write ───────────────────────────────────────────────────────────

def _atomic_write(path: Path, write_fn) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=".tmp_")
    try:
        os.close(fd)
        write_fn(Path(tmp))
        os.replace(tmp, path)
    except Exception:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise


print("Training infrastructure helpers defined.")


In [ ]:
# ── Fast _build_pool_caches (vectorised flat cache + batched kNN) ──────────
#
# The original implementation filters the full DataFrame once per unique
# (merchant_id, year, month) key. For MCC 5411 with ~130 k keys and 131 k
# rows this creates ~17 billion row evaluations and takes 15-30 min just for
# the flat cache.
#
# This replacement precomputes cumulative-sum tables indexed by integer date
# ordinals (year*12 + month) so each flat-pool-mean lookup is O(log n) via
# searchsorted.  The kNN loop also batches all per-date queries into a single
# nn.kneighbors() matrix call instead of one call per merchant.

def _build_pool_caches(
    df: pd.DataFrame,
    scenarios: List[dict],
    cost_type_cols: List[str],
) -> Tuple[Dict[tuple, float], Dict[tuple, float]]:
    unique_keys = {
        (
            s["merchant_id"],
            int(s["context_data"].iloc[-1]["year"]),
            int(s["context_data"].iloc[-1]["month"]),
        )
        for s in scenarios
    }
    print(f"  Building flat pool mean cache ({len(unique_keys):,} keys, vectorised)...")

    # ── Vectorised flat-pool-mean: cumulative sums by date_ord ─────────────
    # df is monthly-level (one row per merchant-month).
    df_ord = (df["year"].values * 12 + df["month"].values).astype(np.int64)

    # Global running total across ALL merchants, indexed by date_ord.
    global_df = (
        pd.DataFrame({"date_ord": df_ord, "log_tpv": df[LOG_TARGET].values})
        .sort_values("date_ord")
    )
    global_df["g_cum_sum"]   = global_df["log_tpv"].cumsum()
    global_df["g_cum_count"] = np.arange(1, len(global_df) + 1, dtype=float)
    # Take the last cumulative value within each date_ord bucket.
    global_agg = global_df.groupby("date_ord")[["g_cum_sum", "g_cum_count"]].last()
    g_date_ords  = global_agg.index.values   # sorted int array
    g_cum_sums   = global_agg["g_cum_sum"].values
    g_cum_counts = global_agg["g_cum_count"].values

    # Per-merchant running total for fast "exclude self" subtraction.
    df_with_ord = df.assign(_date_ord=df_ord).sort_values(["merchant_id", "_date_ord"])
    mid_cum_dict: Dict[int, tuple] = {}
    for mid_val, grp in df_with_ord.groupby("merchant_id", sort=False):
        mid_cum_dict[int(mid_val)] = (
            grp["_date_ord"].values,
            grp[LOG_TARGET].cumsum().values,
            np.arange(1, len(grp) + 1, dtype=float),
        )

    flat_cache: Dict[tuple, float] = {}
    for mid, yr, mo in unique_keys:
        cutoff = yr * 12 + mo
        # Global sum/count up to and including cutoff.
        pos_g = int(np.searchsorted(g_date_ords, cutoff, side="right")) - 1
        if pos_g < 0:
            flat_cache[(mid, yr, mo)] = 0.0
            continue
        g_sum   = g_cum_sums[pos_g]
        g_count = g_cum_counts[pos_g]
        # Subtract this merchant's own contribution.
        mid_data = mid_cum_dict.get(int(mid))
        if mid_data is None:
            m_sum, m_count = 0.0, 0.0
        else:
            dords, csums, ccounts = mid_data
            pos_m = int(np.searchsorted(dords, cutoff, side="right")) - 1
            if pos_m < 0:
                m_sum, m_count = 0.0, 0.0
            else:
                m_sum   = float(csums[pos_m])
                m_count = float(ccounts[pos_m])
        pool_sum   = g_sum   - m_sum
        pool_count = g_count - m_count
        flat_cache[(mid, yr, mo)] = float(pool_sum / pool_count) if pool_count > 0 else 0.0

    # ── kNN pool mean cache (batched per date) ─────────────────────────────
    knn_cache: Dict[tuple, float] = dict(flat_cache)
    if not cost_type_cols:
        print("  No cost_type columns — flat pool mean used for kNN cache.")
        return flat_cache, knn_cache

    keys_by_date: Dict[tuple, list] = defaultdict(list)
    for mid, yr, mo in unique_keys:
        keys_by_date[(yr, mo)].append(mid)

    n_dates = len(keys_by_date)
    print(f"  Building kNN cache ({len(unique_keys):,} keys, {n_dates} dates, k={KNN_K}, batched)...")

    for idx, ((yr, mo), query_mids) in enumerate(sorted(keys_by_date.items())):
        cutoff       = yr * 12 + mo
        snap         = df[df_ord <= cutoff]
        fp_all       = snap.groupby("merchant_id")[cost_type_cols].mean()
        log_tpv_all  = snap.groupby("merchant_id")[LOG_TARGET].mean()

        if len(fp_all) < KNN_K + 1:
            continue

        nn_date = NearestNeighbors(n_neighbors=min(KNN_K + 1, len(fp_all)), metric="cosine")
        nn_date.fit(fp_all.values)
        fp_index = fp_all.index.tolist()

        # Batch: one kneighbors call for all query merchants at this date.
        ctx_rows   = df[(df_ord == cutoff) & df["merchant_id"].isin(set(query_mids))]
        if ctx_rows.empty:
            continue
        query_fp   = ctx_rows.groupby("merchant_id")[cost_type_cols].mean()
        valid_mids = [m for m in query_mids if m in query_fp.index and m in fp_all.index]
        if not valid_mids:
            continue

        _, raw_idxs = nn_date.kneighbors(query_fp.loc[valid_mids].values)
        for i, mid in enumerate(valid_mids):
            top_ids = [fp_index[j] for j in raw_idxs[i] if fp_index[j] != mid][:KNN_K]
            if len(top_ids) == KNN_K:
                knn_cache[(mid, yr, mo)] = float(log_tpv_all.loc[top_ids].mean())

        if (idx + 1) % 20 == 0:
            print(f"    {idx + 1}/{n_dates} dates processed...")

    print("  Pool caches complete.")
    return flat_cache, knn_cache


print("Fast _build_pool_caches defined (vectorised flat cache + batched kNN).")


## Step 4 — Core training function

Trains one `(mcc, context_len)` bundle: HuberRegressor models → conformal calibration → GBR risk models → stratification selection → atomic artifact write.

In [ ]:
def _train_context_len(
    mcc: int,
    ctx_len: int,
    df: pd.DataFrame,
    flat_cache: Dict[tuple, float],
    knn_cache: Dict[tuple, float],
    all_scenarios: List[dict],
    window_start: str,
    window_end: str,
    artifact_base: Path,
) -> None:
    t0 = time.monotonic()
    print(f"\n{'─'*60}")
    print(f"  MCC={mcc}  ctx_len={ctx_len}")
    print(f"{'─'*60}")

    # 1. Merchant-level split
    train_s, val_s, test_s = _merchant_split(all_scenarios)
    print(f"  train={len(train_s):,}  val={len(val_s):,}  test={len(test_s):,}")

    # 2. Temporal conformal split: use last-but-one year as cal, last as test
    ci_years = sorted({int(s["horizon_data"].iloc[0]["year"]) for s in (train_s + val_s + test_s)})
    if len(ci_years) < 2:
        print(f"  SKIP: fewer than 2 horizon years for ctx={ctx_len}")
        return
    cal_year  = ci_years[-2]
    test_year = ci_years[-1]

    train_ci = [s for s in train_s if int(s["horizon_data"].iloc[0]["year"]) < cal_year]
    cal_ci   = [s for s in val_s   if int(s["horizon_data"].iloc[0]["year"]) == cal_year]
    test_ci  = [s for s in test_s  if int(s["horizon_data"].iloc[0]["year"]) == test_year]

    if not train_ci or not cal_ci:
        print(f"  SKIP: empty train_ci or cal_ci for ctx={ctx_len}")
        return

    print(f"  cal_year={cal_year}  test_year={test_year}")
    print(f"  train_ci={len(train_ci):,}  cal_ci={len(cal_ci):,}  test_ci={len(test_ci):,}")

    # 3. Features + HuberRegressor
    y_tr_log  = np.array([s["horizon_data"][LOG_TARGET].values   for s in train_ci])
    y_cal_log = np.array([s["horizon_data"][LOG_TARGET].values   for s in cal_ci])
    y_cal_dol = np.array([s["horizon_data"][TARGET_COL].values   for s in cal_ci])

    X_tr_raw  = _build_feature_matrix(train_ci, knn_cache)
    X_cal_raw = _build_feature_matrix(cal_ci,   knn_cache)

    # Dollar-weighted sample weights: large-volume merchants dominate
    sw = np.array([
        (1.0 / np.log1p(s["context_data"]["transaction_count"].mean()))
        * np.expm1(np.mean(s["context_data"][LOG_TARGET].values))
        for s in train_ci
    ], dtype=float)

    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_tr_raw)
    X_cal = scaler.transform(X_cal_raw)

    models: List[HuberRegressor] = []
    cal_preds_log = np.zeros_like(y_cal_log, dtype=float)
    for h in range(HORIZON_LEN):
        m = HuberRegressor(epsilon=1.35, max_iter=500)
        m.fit(X_tr, y_tr_log[:, h], sample_weight=sw)
        cal_preds_log[:, h] = m.predict(X_cal)
        models.append(m)
        print(f"  h={h+1}  coefs={m.coef_.round(3)}")

    # 4. Dollar calibration residuals + global q90
    cal_preds_dol = np.expm1(cal_preds_log)
    cal_res_dol   = np.abs(y_cal_dol - cal_preds_dol)
    cal_max_res   = cal_res_dol.max(axis=1)
    global_q90    = float(np.quantile(cal_max_res, TARGET_COV))
    print(f"  global_q90 = ±${global_q90:,.2f}  (n_cal={len(cal_max_res):,})")

    cal_mids = np.array([s["merchant_id"] for s in cal_ci])
    cal_residuals: Dict[int, List[float]] = defaultdict(list)
    for i, res in enumerate(cal_max_res):
        cal_residuals[int(cal_mids[i])].append(float(res))
    cal_residuals = dict(cal_residuals)

    # 5. Cross-fitted GBR risk models
    print("  Training GBR risk models (cross-fitted)...")
    train_feat = _build_risk_features(train_ci, flat_cache, knn_cache)
    train_years = np.array([int(s["horizon_data"].iloc[0]["year"]) for s in train_ci])
    fold_cuts   = sorted(set(train_years.tolist()))[1:]
    train_cf_res = np.full((len(train_ci), HORIZON_LEN), np.nan)

    for cut in fold_cuts:
        cf_tr_mask = train_years < cut
        cf_te_mask = train_years == cut
        if cf_tr_mask.sum() == 0 or cf_te_mask.sum() == 0:
            continue
        cf_tr  = [train_ci[j] for j in np.where(cf_tr_mask)[0]]
        cf_te  = [train_ci[j] for j in np.where(cf_te_mask)[0]]
        y_cf_tr_log = np.array([s["horizon_data"][LOG_TARGET].values for s in cf_tr])
        y_cf_te_dol = np.array([s["horizon_data"][TARGET_COL].values for s in cf_te])

        X_cf_tr = _build_feature_matrix(cf_tr, knn_cache)
        X_cf_te = _build_feature_matrix(cf_te, knn_cache)
        sw_cf = np.array([
            (1.0 / np.log1p(s["context_data"]["transaction_count"].mean()))
            * np.expm1(np.mean(s["context_data"][LOG_TARGET].values))
            for s in cf_tr
        ], dtype=float)
        sc_cf    = StandardScaler()
        Xcf_tr   = sc_cf.fit_transform(X_cf_tr)
        Xcf_te   = sc_cf.transform(X_cf_te)

        preds_cf_log = np.zeros((len(cf_te), HORIZON_LEN), dtype=float)
        for h in range(HORIZON_LEN):
            mcf = HuberRegressor(epsilon=1.35, max_iter=500)
            mcf.fit(Xcf_tr, y_cf_tr_log[:, h], sample_weight=sw_cf)
            preds_cf_log[:, h] = mcf.predict(Xcf_te)

        train_cf_res[cf_te_mask] = np.abs(y_cf_te_dol - np.expm1(preds_cf_log))

    cf_valid = np.isfinite(train_cf_res).all(axis=1)
    risk_models = []
    for h in range(HORIZON_LEN):
        gbr = GradientBoostingRegressor(
            loss="squared_error", n_estimators=GBR_N_ESTIMATORS,
            learning_rate=GBR_LEARNING_RATE, max_depth=GBR_MAX_DEPTH,
            min_samples_leaf=GBR_MIN_SAMPLES_LEAF, subsample=GBR_SUBSAMPLE,
            random_state=GBR_RANDOM_STATE,
        )
        y_gbr = np.log1p(train_cf_res[cf_valid, h]) if cf_valid.sum() >= 50 else np.zeros(cf_valid.sum())
        gbr.fit(train_feat[cf_valid], y_gbr)
        risk_models.append(gbr)

    # 6. Stratification scheme selection (leak-free on cal set)
    strat_enabled = False
    strat_scheme  = None
    strat_knot_x  = None
    strat_q_vals  = None

    cal_feat = _build_risk_features(cal_ci, flat_cache, knn_cache)
    cal_risk = np.max(np.column_stack([m.predict(cal_feat) for m in risk_models]), axis=1)
    cal_mids_arr = np.array(sorted(set(cal_mids.tolist())))
    rng  = np.random.default_rng(GBR_RANDOM_STATE)
    perm = rng.permutation(cal_mids_arr)
    cut  = min(max(1, int(round(len(perm) * 0.70))), len(perm) - 1)
    sel_mids  = set(perm[:cut].tolist())
    sel_mask  = np.isin(cal_mids, list(sel_mids))
    eval_mask = ~sel_mask

    if eval_mask.sum() > 0:
        y_eval_dol     = y_cal_dol[eval_mask]
        preds_eval_dol = cal_preds_dol[eval_mask]
        merchant_sel_res = defaultdict(list)
        for mid_v, res in zip(cal_mids[sel_mask], cal_max_res[sel_mask]):
            merchant_sel_res[int(mid_v)].append(float(res))
        global_q_sel = _adaptive_q(cal_max_res[sel_mask].tolist()) or global_q90

        hw_eval_pool = np.zeros(eval_mask.sum(), dtype=float)
        eval_ci_list = [cal_ci[j] for j in np.where(eval_mask)[0]]
        for i, s in enumerate(eval_ci_list):
            yr, mo    = s["context_range"][1]
            peer_ids  = _generate_pool_ids(df, int(s["merchant_id"]), yr, mo)
            peer_res  = [r for pid in peer_ids for r in merchant_sel_res.get(pid, [])]
            q         = _adaptive_q(peer_res) if len(peer_res) >= MIN_POOL else None
            hw_eval_pool[i] = q if q is not None else global_q_sel

        baseline_eval_hw = float(np.mean(hw_eval_pool))
        _min_gain = max(VOL_MIN_GAIN_ABS, VOL_MIN_GAIN_REL * baseline_eval_hw)
        sel_risk  = cal_risk[sel_mask]
        eval_risk = cal_risk[eval_mask]

        scheme_results = []
        for _name, _pct in VOL_BUCKET_SCHEMES.items():
            mapped = _continuous_width_map(sel_risk, eval_risk, cal_max_res[sel_mask], np.array(_pct), global_q_sel)
            if mapped is None:
                continue
            hw     = mapped["hw_apply"]
            lo_    = np.clip(preds_eval_dol - hw[:, None], 0, None)
            hi_    = preds_eval_dol + hw[:, None]
            in_    = (y_eval_dol >= lo_) & (y_eval_dol <= hi_)
            scheme_results.append({
                "name": _name, "pct_edges": np.array(_pct),
                "avg_hw": float(np.mean(hw)),
                "joint_cov": float(np.mean(in_.all(axis=1))),
                "gain": baseline_eval_hw - float(np.mean(hw)),
            })

        feasible = [r for r in scheme_results
                    if r["joint_cov"] >= TARGET_COV and r["gain"] >= _min_gain]
        if feasible:
            best = min(feasible, key=lambda r: (r["avg_hw"], len(r["pct_edges"])))
            mapped_full = _continuous_width_map(cal_risk, cal_risk, cal_max_res, best["pct_edges"], global_q90)
            if mapped_full is not None:
                strat_enabled = True
                strat_scheme  = best["name"]
                strat_knot_x  = mapped_full["knot_x"]
                strat_q_vals  = mapped_full["q_vals"]
                print(f"  Stratification PASSED: scheme={strat_scheme}")
            else:
                print("  Stratification failed on full calibration set")
        else:
            print(f"  No feasible stratification scheme (tested {len(scheme_results)})")

    # 7. Atomic artifact write
    artifact_dir = artifact_base / str(mcc) / str(ctx_len)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    _atomic_write(artifact_dir / "models.pkl",       lambda p: joblib.dump(models,       p))
    _atomic_write(artifact_dir / "scaler.pkl",       lambda p: joblib.dump(scaler,       p))
    _atomic_write(artifact_dir / "cal_residuals.pkl",lambda p: joblib.dump(cal_residuals,p))
    _atomic_write(artifact_dir / "global_q90.pkl",   lambda p: joblib.dump(global_q90,  p))
    _atomic_write(artifact_dir / "risk_models.pkl",  lambda p: joblib.dump(risk_models, p))

    if strat_enabled and strat_knot_x is not None:
        _atomic_write(artifact_dir / "strat_knot_x.pkl", lambda p: joblib.dump(strat_knot_x, p))
        _atomic_write(artifact_dir / "strat_q_vals.pkl", lambda p: joblib.dump(strat_q_vals, p))

    snapshot = {
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "mcc": mcc,
        "context_len": ctx_len,
        "horizon_len": HORIZON_LEN,
        "window_start": window_start,
        "window_end": window_end,
        "n_train_ci": len(train_ci),
        "n_cal_ci": len(cal_ci),
        "n_test_ci": len(test_ci) if test_ci else 0,
        "cal_year": cal_year,
        "global_q90_dollars": global_q90,
        "knn_k": KNN_K,
        "target_cov": TARGET_COV,
        "strat_enabled": strat_enabled,
        "strat_scheme": strat_scheme,
        "n_model_features": 11,
        "n_risk_features": 11,
        "conformal_space": "dollar",
        "bias_correction": "none",
        "sample_weighting": "dollar_weighted",
        "data_source": "processed_transactions_4mcc.csv",
        "use_full_data": USE_FULL_DATA,
    }
    _atomic_write(artifact_dir / "config_snapshot.json",
                  lambda p: p.write_text(json.dumps(snapshot, indent=2)))

    elapsed = time.monotonic() - t0
    print(f"  Artifacts saved → {artifact_dir}  [{elapsed:.1f}s]")


print("Core training function defined.")

## Step 5 — Run training for all MCCs

This cell is the long-running one. For the full 4-MCC dataset it takes ~20–60 min.  
Artifacts are written incrementally — each `(mcc, ctx_len)` bundle is complete before the next starts.  
The live `ml_service` container will hot-reload each bundle within 60 s of the `config_snapshot.json` mtime changing.

In [ ]:
t_total = time.monotonic()

for mcc in TRAIN_MCCS:
    print(f"\n{'='*60}")
    print(f"  Training MCC {mcc}")
    print(f"{'='*60}")

    # Skip MCCs that already have complete artifacts from a prior run.
    # Set FORCE_RETRAIN = True in the config cell to override.
    if not FORCE_RETRAIN and all(
        (ARTIFACTS_OUTPUT_PATH / str(mcc) / str(ctx_len) / "config_snapshot.json").exists()
        for ctx_len in SUPPORTED_CONTEXT_LENS
    ):
        print(f"  SKIP MCC {mcc}: all artifacts already exist (set FORCE_RETRAIN=True to override)")
        continue

    df = monthly_by_mcc[mcc].copy()

    # ── Window selection ────────────────────────────────────────────────
    if USE_FULL_DATA:
        window_start = f"{int(df['year'].min())}-{int(df['month'].min()):02d}"
        window_end   = f"{int(df['year'].max())}-{int(df['month'].max()):02d}"
        print(f"  Using full data window: {window_start} → {window_end}")
    else:
        import datetime as _dt
        today = _dt.date.today()
        cutoff_year = today.year - 3
        df = df[(df["year"] > cutoff_year) |
                ((df["year"] == cutoff_year) & (df["month"] >= today.month))]
        if df.empty:
            print(f"  SKIP MCC {mcc}: no data in rolling 3-year window — set USE_FULL_DATA=True")
            continue
        window_start = f"{int(df['year'].min())}-{int(df['month'].min()):02d}"
        window_end   = f"{int(df['year'].max())}-{int(df['month'].max()):02d}"
        print(f"  Rolling window: {window_start} → {window_end}")

    print(f"  {len(df):,} merchant-months, {df['merchant_id'].nunique():,} merchants")

    # ── Build scenarios ─────────────────────────────────────────────────
    print(f"\n  Building scenarios for context lengths {SUPPORTED_CONTEXT_LENS}...")
    scenarios_by_ctx: Dict[int, List[dict]] = {}
    all_scenarios_combined: List[dict] = []
    for ctx_len in SUPPORTED_CONTEXT_LENS:
        scens = _build_scenarios(df, ctx_len)
        scenarios_by_ctx[ctx_len] = scens
        all_scenarios_combined.extend(scens)
        n_m = len({s["merchant_id"] for s in scens})
        print(f"  ctx={ctx_len}: {len(scens):,} scenarios from {n_m:,} merchants")

    if not all_scenarios_combined:
        print(f"  SKIP MCC {mcc}: no scenarios generated")
        continue

    # ── Build pool caches ───────────────────────────────────────────────
    print(f"\n  Building pool caches...")
    ct_cols_present = [c for c in df.columns if c.startswith("cost_type_") and c.endswith("_pct")]
    flat_cache, knn_cache = _build_pool_caches(df, all_scenarios_combined, ct_cols_present)

    # ── Train per context length ────────────────────────────────────────
    for ctx_len in SUPPORTED_CONTEXT_LENS:
        scens = scenarios_by_ctx[ctx_len]
        if len(scens) < 50:
            print(f"\n  SKIP ctx_len={ctx_len}: only {len(scens)} scenarios")
            continue
        _train_context_len(
            mcc=mcc,
            ctx_len=ctx_len,
            df=df,
            flat_cache=flat_cache,
            knn_cache=knn_cache,
            all_scenarios=scens,
            window_start=window_start,
            window_end=window_end,
            artifact_base=ARTIFACTS_OUTPUT_PATH,
        )

print(f"\n{'='*60}")
print(f"ALL MCCs DONE in {(time.monotonic()-t_total)/60:.1f} min")
print(f"Artifacts written to: {ARTIFACTS_OUTPUT_PATH}")
print(f"{'='*60}")


## Step 6 — Verify artifacts

Check all expected artifact files exist and print the `global_q90` for each bundle.  
A healthy value for production-scale merchants is typically **$5,000–$100,000** depending on merchant size.  
The old value was **~$296** (trained on synthetic composite merchants).

In [ ]:
REQUIRED_ARTIFACTS = [
    "models.pkl", "scaler.pkl", "cal_residuals.pkl",
    "global_q90.pkl", "risk_models.pkl", "config_snapshot.json",
]

print(f"{'MCC':<8} {'ctx':<6} {'global_q90':>14} {'strat':>8} {'n_cal':>8}  status")
print("-" * 58)

all_ok = True
for mcc in TRAIN_MCCS:
    for ctx_len in SUPPORTED_CONTEXT_LENS:
        artifact_dir = ARTIFACTS_OUTPUT_PATH / str(mcc) / str(ctx_len)
        if not artifact_dir.exists():
            print(f"{mcc:<8} {ctx_len:<6} {'—':>14} {'—':>8} {'—':>8}  MISSING DIR")
            all_ok = False
            continue
        missing = [f for f in REQUIRED_ARTIFACTS if not (artifact_dir / f).exists()]
        if missing:
            print(f"{mcc:<8} {ctx_len:<6} {'—':>14} {'—':>8} {'—':>8}  MISSING: {missing}")
            all_ok = False
            continue
        snap = json.loads((artifact_dir / "config_snapshot.json").read_text())
        q90    = snap.get("global_q90_dollars", 0)
        strat  = "yes" if snap.get("strat_enabled") else "no"
        n_cal  = snap.get("n_cal_ci", "?")
        print(f"{mcc:<8} {ctx_len:<6} ${q90:>13,.0f} {strat:>8} {n_cal:>8}  OK")

print("-" * 58)
if all_ok:
    print("\nAll artifacts present. The ml_service will hot-reload within 60 s.")
else:
    print("\nSome bundles missing — re-run Step 5 for those MCCs.")

## Step 7 — Update ml_service SUPPORTED_MCCS

After training, the ml_service `tpv_forecast/service.py` needs `SUPPORTED_MCCS` extended to include the new MCCs so the `GetTPVForecast` endpoint uses Huber inference instead of falling back to SARIMA.

Run the cell below — it patches the file in place.

In [ ]:
import re

# SUPPORTED_MCCS now lives in config.py, not service.py
config_path = REPO_ROOT / "ml_service" / "modules" / "tpv_forecast" / "config.py"

# Only update entries for MCCs that actually have trained artifacts
trained_mccs = sorted([
    mcc for mcc in TRAIN_MCCS
    if all(
        (ARTIFACTS_OUTPUT_PATH / str(mcc) / str(ctx) / "config_snapshot.json").exists()
        for ctx in SUPPORTED_CONTEXT_LENS
    )
])
print(f"MCCs with complete artifacts: {trained_mccs}")

if not trained_mccs:
    print("No complete artifact sets found — run Step 5 first.")
else:
    src = config_path.read_text(encoding="utf-8")

    # Matches both bare  SUPPORTED_MCCS = [...]  and annotated  SUPPORTED_MCCS: list[int] = [...]
    old_line = re.search(r"SUPPORTED_MCCS\s*(?::\s*\S+\s*)?=\s*\[.*?\]", src)
    if old_line:
        new_line = f"SUPPORTED_MCCS: list[int] = {trained_mccs}"
        updated  = src[:old_line.start()] + new_line + src[old_line.end():]
        config_path.write_text(updated, encoding="utf-8")
        print(f"Updated SUPPORTED_MCCS → {trained_mccs} in {config_path}")
    else:
        print(f"WARNING: Could not find SUPPORTED_MCCS in {config_path} — update manually.")

print("\nDone. Rebuild/restart the ml_service container to apply.")


## Step 8 — Switch ml_service to direct inference

The current `run_tpv_forecast` uses a **centered-delta workaround** that was needed because old artifacts were trained on ~$100/month synthetic merchants.  
New artifacts are trained on real production merchants — direct `scaler.transform(X_raw)` is correct.

This cell patches `ml_service/modules/tpv_forecast/service.py` to remove the centering hack and use direct inference, matching the Matt_EDA v2 reference implementation.

In [ ]:
service_path = REPO_ROOT / "ml_service" / "modules" / "tpv_forecast" / "service.py"
src = service_path.read_text(encoding="utf-8")

if "Centered-delta inference" in src:
    print("WARNING: The centered-delta workaround is still present in service.py.")
    print("The merge conflict resolution should have removed it.")
    print("Check service.py manually and ensure it uses direct bundle.scaler.transform(X_raw).")
elif "bundle.scaler.transform" in src or "scaler.transform" in src:
    print("OK: service.py uses direct scaler.transform inference (no centering workaround).")
    print("No patch needed — the resolved service.py matches the v2 reference implementation.")
    print("New artifacts (trained on production-scale data) will be used directly.")
else:
    print("Could not verify inference method in service.py — check manually.")
    print("Expected to find 'scaler.transform' in the inference path.")
